In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:
# Task 1: Write your code here:
# Load the dataset
df_path=os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(df_path)

print(f"Dataset shape: {df.shape}")


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
################### ارجع
y= df["Delivery_Time"].value_counts()
distance=df["Distance_km"].value_counts()


plt.scatter(y.index,y)

In [ ]:
# Task 1: Write your code here:
df.drop(columns="Order_ID")

In [ ]:
df.info()


In [ ]:

missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(200)

In [ ]:
# Task 2: Write your code here:
#Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )

df.dropna(subset=['Delivery_Time','Weather','Traffic_Level',"Time_of_Day","Courier_Experience_yrs"])


In [ ]:
# Task 3: Write your code here:
#Check and remove duplicates if any exist
def check_duplicates(df):

  #TODO: get duplicated data using pandas
  duplicates = df.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
#Encode categorical variables if needed (Bonus if used One Hot Encoding)
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

categorical_cols = df.select_dtypes(include=["object"]).columns
df['Delivery_Time'] = df['Delivery_Time'].fillna(0)

print("Categorical Columns:", list(categorical_cols))
le = LabelEncoder()

# Encode the target column
df["Weather"] = le.fit_transform(df["Weather"])
df["Traffic_Level"] = le.fit_transform(df["Traffic_Level"])
df["Time_of_Day"] = le.fit_transform(df["Time_of_Day"])
df["Vehicle_Type"] = le.fit_transform(df["Vehicle_Type"])

df

In [ ]:
# Task 5: Write your code here:
#Apply feature scaling for all features (Use StandardScaler)
from sklearn.preprocessing import StandardScaler #import StandardScaler


standard_scaler = StandardScaler() # Instantiate StandardScaler
data_standard_scaled = standard_scaler.fit_transform(df) # Apply fit_transform

print('\nData after scaling:\n', data_standard_scaled) #show after scaling

In [ ]:
# Task 6: Write your code here:
#Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df["Delivery_Time"].astype(float)
df['Delivery_Time'] = df['Delivery_Time'].fillna('none')
y.describe()

In [ ]:
# Task 2,3,4,5: Write your code here:
#Use the correct split: KFold OR StratifiedKFold
#Train a RandomForest model
#Evaluate using MAE (Mean Absolute Error) ONLY
#Print the averaged score across all folds
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier

""""Random Forest": RandomForestClassifier(
      n_estimators=320,  # Number of trees
      max_depth=4"""
MEA=[]
n_splits = 5 # K
model = RandomForestClassifier(
      n_estimators=320,  # Number of trees
      max_depth=4)
# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate
  mae = mean_absolute_error(y_test, y_pred)
  MEA.append(mae)
avg_MEA=0
for i in MEA:
  avg_MEA+=i
print("the averaged score across all folds is :",avg_MEA/n_splits)

In [ ]:
df.info()

In [ ]:
# Task 1: Write your code here:
feature_cols =   ['Order_ID', 'Distance_km', 'Weather', 'Traffic_Level',"Time_of_Day","Vehicle_Type","Preparation_Time_min","Courier_Experience_yrs"]
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()



In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()
#note the 0 mean nan

In [ ]:
# Task Bonus: Write your code here: